In [1]:
#-----------------------------
# Model : miniCrossNet-Hv0
# Written by : Akash Lanjhi
# Contact : akashl@iitk.ac.in
#-----------------------------

# Importing dependency
import time
import math
import random
import numpy as np
import scipy.io as sio

import torch
import torch.nn as nn
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ExponentialLR, CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

from thop import profile
from torchinfo import summary

from colorama import Fore, Back, Style
from tqdm.notebook import tqdm_notebook

import warnings
warnings.filterwarnings("ignore")
print(f'{Fore.BLUE}{Style.BRIGHT}- Dependency:{Fore.RESET}', end=' ')
print(f'{Fore.GREEN}Done{Fore.RESET}{Style.RESET_ALL}')

- Dependency: Done


In [ ]:
#-------------
# Random Seed
#-------------
seed = 2025
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [3]:
#------------------
# Computing Device 
#------------------

# Device
device = torch.device("cpu")
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda:1")
    
print(f'{Fore.BLUE}{Style.BRIGHT}- Compute Device:{Fore.RESET}', end=' ')
print(f'{Fore.GREEN}{str(device).upper()}{Fore.RESET}{Style.RESET_ALL}')

- Compute Device: CUDA:1


In [4]:
#---------------------
# Important Functions
#---------------------

# Model Info
def TorchinfoSummary(model, input_data, device):
    """
    Summarize the given PyTorch model
    Args:
        model (nn.Module): pytorch model
        input_data (torch.Tensor): sample input to the model
        device (str): compute device

    Returns:
        None : return nothing
    """
    print(f"{Fore.YELLOW}{summary(model=model, input_data=input_data, verbose=False, device=device)}")
    
    return None


# Thop Params and Flops Calculation
def ThopParamsFlops(model, input_data, device):
    """
    Calculate the parameters and flops of the given PyTorch model
    Args:
        model (nn.Module): pytorch model
        input_data (torch.Tensor): sample input to the model
        device (str): compute device

    Returns:
        None: return nothing
    """
    Flops, Params = profile(model=model, inputs=(input_data.to(device), ), verbose=False)
    print(f"{Fore.MAGENTA}- Toltal FLOPS: {Fore.CYAN}{Flops/1e6}M")
    print(f"{Fore.MAGENTA}- Total Params: {Fore.CYAN}{Params/1e6}M")
    
    return None

    
# NMSE Calculation
def NMSE(predicted_channel, actual_channel):
    """
    Calculate the normalized mean square error of the given tensors
    Args:
        predicted_channel (torch.Tensor): reconstructed channel by the model 
        actual_channel (torch.Tensor): actual ground truth channel

    Returns:
        float: normalized mean square error
    """
    # Numpy arrays to PyTorch tensor
    if isinstance(actual_channel, np.ndarray):
        actual_channel = torch.tensor(actual_channel, dtype=torch.float32) 
    if isinstance(predicted_channel, np.ndarray):
        predicted_channel = torch.tensor(predicted_channel, dtype=torch.float32)   
    
    # Moving to GPU    
    actual_channel = actual_channel.to(device)
    predicted_channel = predicted_channel.to(device)
    
    with torch.no_grad():
        # De-centralize
        actual_channel = actual_channel - 0.5
        predicted_channel = predicted_channel - 0.5
        # NMSE Calculation
        power = actual_channel[:, 0, :, :] ** 2 + actual_channel[:, 1, :, :] ** 2
        difference = actual_channel - predicted_channel
        mse = difference[:, 0, :, :] ** 2 + difference[:, 1, :, :] ** 2
        nmse = 10 * torch.log10((mse.sum(dim=(1, 2)) / power.sum(dim=(1, 2))).mean())
        
        return float(nmse)


# RHO Calculation
def RHO(actual_channel, predicted_channel):
    """
    Calculate the cosine similarity
    Args:
        actual_channel (torch.Tensor): actual ground truth channel
        predicted_channel (torch.Tensor): reconstructed channel by the model

    Returns:
        float: rho
    """
    actual_channel_real = actual_channel[:, 0, :, :]
    actual_channel_imag = actual_channel[:, 1, :, :]
    actual_channel_comp = (actual_channel_real - 0.5) + 1j * (actual_channel_imag - 0.5)

    predicted_channel_real = predicted_channel[:, 0, :, :]
    predicted_channel_imag = predicted_channel[:, 1, :, :]
    predicted_channel_comp = (predicted_channel_real - 0.5) + 1j * (predicted_channel_imag - 0.5)

    n1 = (torch.sum(torch.conj(actual_channel_comp)*actual_channel_comp, axis=2))
    n1 = n1.type(torch.DoubleTensor)
    n1 = n1.to(device)
    n2 = (torch.sum(torch.conj(predicted_channel_comp)*predicted_channel_comp, axis=2))
    n2 = n2.type(torch.DoubleTensor)
    n2 = n2.to(device)
    aa = torch.square(abs(torch.sum(torch.conj(actual_channel_comp)*predicted_channel_comp, axis=2)))
    rho = torch.mean(torch.mean(aa/(n1*n2), axis=1))
 
    return rho

In [5]:
#---------------
# Dataset Class
#---------------

# Dataset Class For Case0 [H1] --> [H1]
class CustomDataset(Dataset):
    def __init__(self, path:str):
        self.mat = sio.loadmat(path)['HT']
        self.x = np.reshape(self.mat, (self.mat.shape[0], 2, 32, 32))
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        samplex = self.x[idx]
        return torch.tensor(samplex, dtype=torch.float32)

In [6]:
#------------
# DataLoader 
#------------

# Dataset
torch.manual_seed(seed)
train_data = CustomDataset("/home/subhodeep/swin/data/COST2100/Indoor/DATA_Htrainin.mat")
val_data = CustomDataset("/home/subhodeep/swin/data/COST2100/Indoor/DATA_Hvalin.mat")
test_data = CustomDataset("/home/subhodeep/swin/data/COST2100/Indoor/DATA_Htestin.mat")

# DataLoader
torch.manual_seed(seed)
train_loader = DataLoader(train_data, batch_size=200, shuffle=True, pin_memory=True, pin_memory_device="cuda:1")
val_loader = DataLoader(val_data, batch_size=200, shuffle=False, pin_memory=True, pin_memory_device="cuda:1")
test_loader = DataLoader(test_data, batch_size=200, shuffle=False, pin_memory=True, pin_memory_device="cuda:1")

print(f'{Fore.BLUE}{Style.BRIGHT}- DataLoader:{Fore.RESET}', end=' ')
print(f'{Fore.GREEN}Created Suceesfully{Fore.RESET}{Style.RESET_ALL}')

- DataLoader: Created Suceesfully


In [7]:
#---------------
# Model Modules
#---------------

# Multi Layer Perceptron
class MLP(nn.Module):
    """
    input:  (200, 32, 64)
    output: (200, 32, 64)
    
    Args:
        dim (int): dimension of the model
    """
    def __init__(self, dim=64):
        super(MLP, self).__init__()
        self.dim = dim
        self.fc1 = nn.Linear(in_features=dim, out_features=4*dim)
        self.fc2 = nn.Linear(in_features=4*dim, out_features=dim)
        self.act = nn.GELU()
        
    # forward pass    
    def forward(self, x):
        B, N, C = x.shape           # (B, N, C)
        x = self.fc1(x)             # (B, N, 8*C)
        x = self.act(x)             # (B, N, 8*C)
        x = self.fc2(x)             # (B, N, C)
        
        return x                    # (B, N, C)


# Query Key Value
class QKV(nn.Module):
    """
    input:  (200, 32, 64)
    output: (200, 32, 64)
    
    Args:
        dim (int): dimension of the model
        heads (int): number of heads
    """
    def __init__(self, dim=64, heads=4):
        super(QKV, self).__init__()
        self.dim = dim
        self.heads = heads
        self.w_q = nn.Linear(dim, dim//2, bias=True)
        self.w_k = nn.Linear(dim, dim//2, bias=True)
        self.w_v = nn.Linear(dim, dim//2, bias=True)
        
    # forward pass    
    def forward(self, x):
        B, N, C = x.shape                                                                                   # (B, N, C)
        q = self.w_q(x).reshape(B, N, self.heads, C//(2*self.heads)).permute(0, 2, 1, 3).contiguous()       # (B, N, C) --> (B, N, Heads, dim/heads) --> (B, Heads, N, dim/heads)
        k = self.w_k(x).reshape(B, N, self.heads, C//(2*self.heads)).permute(0, 2, 1, 3).contiguous()       # (B, N, C) --> (B, N, Heads, dim/heads) --> (B, Heads, N, dim/heads)
        v = self.w_v(x).reshape(B, N, self.heads, C//(2*self.heads)).permute(0, 2, 1, 3).contiguous()       # (B, N, C) --> (B, N, Heads, dim/heads) --> (B, Heads, N, dim/heads)
        
        return q, k, v                                                                                      # (B, Heads, N, dim/heads)


# Multi Head Attention
class MHA(nn.Module):
    """
    input:  (200, 32, 64)
    output: (200, 32, 64)
    
    Args:
        dim (int): dimension of the model
        heads (int): number of heads
    """
    def __init__(self, dim=64, heads=4):
        super(MHA, self).__init__()
        self.dim = dim
        self.heads = heads
        self.scale = (dim // (2*heads)) ** -0.5
        self.proj = nn.Linear(dim//2, dim)
        
    # forward pass    
    def forward(self, query, key, value):
        B, H, N, D = query.shape                                            # (B, Heads, N, dim/heads)
        attn = (query @ key.transpose(-2, -1)) * self.scale                 # (B, Heads, N, N)
        attn = attn.softmax(dim=-1)                                         # (B, Heads, N, N) 
        attn_out = (attn @ value).transpose(1, 2).reshape(B, N, H*D)        # (B, Heads, N, dim/heads) --> (B, N, Heads, dim/heads) --> (B, N, dim)
        out = self.proj(attn_out)                                           # (B, N, dim)
        
        return out                                                          # (B, N, dim)    


# Encoder
class Encoder(nn.Module):
    """
    input:  (200, 32, 64)
    output: (200, 32, 64)
    
    Args:
        dim (int): dimension of the model
        heads (int): number of heads in mha
    """
    def __init__(self, dim=64, heads=4):
        super(Encoder, self).__init__()
        self.dim = dim
        self.heads = heads
        self.norm1 = nn.LayerNorm(dim)
        self.qkv = QKV(dim=dim, heads=heads)
        self.mha = MHA(dim=dim, heads=heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim=dim)
    
    # forward pass    
    def forward(self, x):
        B, N, C = x.shape                      # (B, N, C)
        Q, K, V = self.qkv(self.norm1(x))      # (B, N, C)
        x = x + self.mha(Q, K, V)              # (B, N, C)
        x = x + self.mlp(self.norm2(x))        # (B, N, C)
        
        return x                               # (B, N, C)


# Cross Encoder
class CrossEncoder(nn.Module):
    """
    input:  (200, 32, 64)
    output: (200, 32, 64)
    
    Args:
        dim (int): dimension of the model
        heads (int): number of heads in mha
    """
    def __init__(self, dim=64, heads=4):
        super(CrossEncoder, self).__init__()
        self.dim = dim
        self.heads = heads
        # Antenna to Subcarrier Attention
        self.norm1 = nn.LayerNorm(dim)
        self.qkv1 = QKV(dim=dim, heads=heads)
        self.mha1 = MHA(dim=dim, heads=heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = MLP(dim=dim)
        # Subcarrier to Antenna Attention
        self.norm3 = nn.LayerNorm(dim)
        self.qkv2 = QKV(dim=dim, heads=heads)
        self.mha2 = MHA(dim=dim, heads=heads)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = MLP(dim=dim)
        
    # forward pass    
    def forward(self, x, y):
        B, N, C = x.shape                          # (B, N, C)
        B, N, C = y.shape                          # (B, N, C)
        # Attention Calculation
        Qx, Kx, Vx = self.qkv1(self.norm1(x))      # (B, N, C)
        Qy, Ky, Vy = self.qkv2(self.norm3(y))      # (B, N, C)
        # Multi-Head Attention
        x = x + self.mha1(Qx, Ky, Vy)              # (B, N, C)
        y = y + self.mha2(Qy, Kx, Vx)              # (B, N, C)
        # Multi-Layer Perceptron
        x = x + self.mlp1(self.norm2(x))           # (B, N, C)
        y = y + self.mlp2(self.norm4(y))           # (B, N, C)
        
        return x, y                                # ((B, N, C), (B, N, C))    


# Decoder
class Decoder(nn.Module):
    """
    input:  (200, 32, 64)
    output: (200, 32, 64)
    
    Args:
        dim (int): dimension of the model
        heads (int): number of heads in mha
    """
    def __init__(self, dim=64, heads=4):
        super(Decoder, self).__init__()
        self.dim = dim
        self.heads = heads
        self.norm1 = nn.LayerNorm(dim)
        self.qkv = QKV(dim=dim, heads=heads)
        self.mha = MHA(dim=dim, heads=heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim=dim)
    
    # forward pass    
    def forward(self, x):
        B, N, C = x.shape                      # (B, N, C)
        Q, K, V = self.qkv(self.norm1(x))      # (B, N, C)
        x = x + self.mha(Q, K, V)              # (B, N, C)
        x = x + self.mlp(self.norm2(x))        # (B, N, C)
        
        return x                               # (B, N, C)


# Cross Encoder
class CrossDecoder(nn.Module):
    """
    input:  (200, 32, 64)
    output: (200, 32, 64)
    
    Args:
        dim (int): dimension of the model
        heads (int): number of heads in mha
    """
    def __init__(self, dim=64, heads=4):
        super(CrossDecoder, self).__init__()
        self.dim = dim
        self.heads = heads
        # Antenna to Subcarrier Attention
        self.norm1 = nn.LayerNorm(dim)
        self.qkv1 = QKV(dim=dim, heads=heads)
        self.mha1 = MHA(dim=dim, heads=heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = MLP(dim=dim)
        # Subcarrier to Antenna Attention
        self.norm3 = nn.LayerNorm(dim)
        self.qkv2 = QKV(dim=dim, heads=heads)
        self.mha2 = MHA(dim=dim, heads=heads)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = MLP(dim=dim)
        
    # forward pass    
    def forward(self, x, y):
        B, N, C = x.shape                          # (B, N, C)
        B, N, C = y.shape                          # (B, N, C)
        # Attention Calculation
        Qx, Kx, Vx = self.qkv1(self.norm1(x))      # (B, N, C)
        Qy, Ky, Vy = self.qkv2(self.norm3(y))      # (B, N, C)
        # Multi-Head Attention
        x = x + self.mha1(Qx, Ky, Vy)              # (B, N, C)
        y = y + self.mha2(Qy, Kx, Vx)              # (B, N, C)
        # Multi-Layer Perceptron
        x = x + self.mlp1(self.norm2(x))           # (B, N, C)
        y = y + self.mlp2(self.norm4(y))           # (B, N, C)
        
        return x, y                                # ((B, N, C), (B, N, C)) 


# Model
class miniCrossNet(nn.Module):
    """
    input:  (200, 2, 32, 32)
    output: (200, 2, 32, 32)
    
    Args:
        seq_len (int): number of input tokens
        dim (int): dimension of the model
        heads (int): number of heads in mha
        tx_codeword (int): transmit codeword
    """
    def __init__(self, seq=32, dim=64, heads=4, codeword=512):
        super(miniCrossNet, self).__init__()
        self.seq = seq
        self.dim = dim
        self.d_model = dim//2
        self.heads = heads
        self.codeword = codeword
        ## Encoder ##
        # Subcarrier
        self.encsubemb = nn.Linear(in_features=self.dim, out_features=self.d_model)
        self.module1 = Encoder(dim=self.d_model, heads=self.heads)
        # Antenna
        self.encantemb = nn.Linear(in_features=self.dim, out_features=self.d_model)
        self.module2 = Encoder(dim=self.d_model, heads=self.heads)
        # Subcarrier-Antenna
        self.module3 = CrossEncoder(dim=self.d_model, heads=self.heads)
        self.fusenorm1 = nn.LayerNorm(self.d_model)
        self.reduction = nn.Linear(self.seq*self.d_model, self.codeword)
        
        ## Decoder ##
        self.expansion = nn.Linear(self.codeword, self.seq*self.dim)
        # Subcarrier
        self.decsubemb = nn.Linear(in_features=self.dim, out_features=self.d_model)
        self.module4 = Decoder(dim=self.d_model, heads=self.heads)
        # Antenna
        self.decantemb = nn.Linear(in_features=self.dim, out_features=self.d_model)
        self.module5 = Decoder(dim=self.d_model, heads=self.heads)
        # Subcarrier-Antenna
        self.module6 = CrossDecoder(dim=self.d_model, heads=self.heads)
        self.fusenorm2 = nn.LayerNorm(self.d_model)
        
        ## Output Head ##
        self.outhead = nn.Linear(in_features=self.d_model, out_features=self.dim)
        self.sigmoid = nn.Sigmoid()
        
    # forward pass    
    def forward(self, x):
        ## Encoder ##
        B, C, H, W = x.shape                                                                                    # (B, C, H, W)
        # Subcarrier
        subcarrier = torch.concat([x[:, 0, :, :], x[:, 1, :, :]], dim=-1)                                       # (B, H, 2*W)
        subcarrier = self.encsubemb(subcarrier)
        subcarrier_attn = self.module1(subcarrier)                                                              # (B, H, d_model)
        # Antenna
        antenna = torch.concat([x.transpose(-2, -1)[:, 0, :, :], x.transpose(-2, -1)[:, 1, :, :]], dim=-1)      # (B, W, 2*H)
        antenna = self.encantemb(antenna)
        antenna_attn = self.module2(antenna)                                                                    # (B, W, d_model)
        # Subcarrier-Antenna
        subcarrier_attn, antenna_attn = self.module3(subcarrier_attn, antenna_attn)                             # (B, H, d_model)
        # Sum Fusion
        x = subcarrier_attn + antenna_attn                                                                      # (B, H, d_model)
        x = self.fusenorm1(x)                                                                                   # (B, H, d_model)
        x = x.reshape(B, -1)                                                                                    # (B, H*d_model)
        x = self.reduction(x)                                                                                   # (B, codeword)
        
        ## Decoder ##
        x = self.expansion(x)                                                                                   # (B, C*H*W)
        x = x.reshape(B, C, H, W)                                                                               # (B, C, H, W)
        # Subcarrier
        subcarrier = torch.concat([x[:, 0, :, :], x[:, 1, :, :]], dim=-1)                                       # (B, H, 2*W)
        subcarrier = self.decsubemb(subcarrier)
        subcarrier_attn = self.module4(subcarrier)                                                              # (B, H, d_model)
        # Antenna
        antenna = torch.concat([x.transpose(-2, -1)[:, 0, :, :], x.transpose(-2, -1)[:, 1, :, :]], dim=-1)      # (B, W, 2*H)
        antenna = self.decantemb(antenna)
        antenna_attn = self.module5(antenna)                                                                    # (B, W, d_model)
        # Subcarrier-Antenna
        subcarrier_attn, antenna_attn = self.module6(subcarrier_attn, antenna_attn)                        # (B, H, d_model)
        # Sum Fusion
        x = subcarrier_attn + antenna_attn                                                                      # (B, H, d_model)                                                                      
        x = self.fusenorm2(x)                                                                                   # (B, H, d_model)
        
        ## Output Head ##
        x = self.outhead(x)
        x = self.sigmoid(x)                                                                                     # (B, H, 2*W)
        out = torch.stack((x[:, :, 0:self.dim//2], x[:, :, self.dim//2:]), dim=1)                               # (B, 2, H, W)
        
        return out                                                                                              # (B, 2, H, W)

In [8]:

# Model
torch.manual_seed(seed)
model = miniCrossNet(seq=32, dim=64, heads=4, codeword=512).to(device)

# Loss
criterion = nn.MSELoss().to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-3, betas=(0.8, 0.98), weight_decay=0.001)

# Scheduler
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=1000, eta_min=5e-5, last_epoch=-1)

In [9]:
# Model info
input_data = torch.randn(1, 2, 32, 32)
TorchinfoSummary(model=model, input_data=input_data, device=device)

Layer (type:depth-idx)                   Output Shape              Param #
miniCrossNet                             [1, 2, 32, 32]            --
├─Linear: 1-1                            [1, 32, 32]               2,080
├─Encoder: 1-2                           [1, 32, 32]               --
│    └─LayerNorm: 2-1                    [1, 32, 32]               64
│    └─QKV: 2-2                          [1, 4, 32, 4]             --
│    │    └─Linear: 3-1                  [1, 32, 16]               528
│    │    └─Linear: 3-2                  [1, 32, 16]               528
│    │    └─Linear: 3-3                  [1, 32, 16]               528
│    └─MHA: 2-3                          [1, 32, 32]               --
│    │    └─Linear: 3-4                  [1, 32, 32]               544
│    └─LayerNorm: 2-4                    [1, 32, 32]               64
│    └─MLP: 2-5                          [1, 32, 32]               --
│    │    └─Linear: 3-5                  [1, 32, 128]              4,224
│    

In [10]:
# Model params and flops
input_data = torch.randn(1, 2, 32, 32)
ThopParamsFlops(model=model, input_data=input_data, device=device)

- Toltal FLOPS: 4.595712M
- Total Params: 1.670848M


In [11]:

# Model Training and Validation
num_epochs = 1000
train_losses = []
val_losses = []
nmse_score = []

for epoch in tqdm_notebook(range(num_epochs), desc="Model Training", colour="#b53fd3", leave=True):
    # Model Training
    model.train()
    train_loss = 0
    for data in tqdm_notebook(train_loader, "Mini Batch Training", colour="#0099ff", leave=False):
        x = data.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        
    # Scheduler   
    scheduler.step()   
        
    if (epoch+1)%1==0:
        # Model Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data in tqdm_notebook(val_loader, "Validating The Model", colour="#0099ff", leave=False):
                x = data.to(device)
                output = model(x)
                loss = criterion(output, x)
                val_loss += loss.item() * x.size(0)     
                
        # Model Testing
        model.eval()
        nmse_error = 0
        with torch.no_grad():
            for data in tqdm_notebook(test_loader, "Calculating NMSE", colour="#0099ff", leave=False):
                x = data.to(device)
                output = model(x)
                nmse = NMSE(output, x) 
                nmse_error+=nmse
                
        # Training Loss
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)
        # Validation Loss
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)
        # NMSE Score
        avg_nmse = nmse_error/len(test_loader)
        nmse_score.append(avg_nmse) 
        
        # Printing Training Details
        print(Fore.LIGHTBLUE_EX + f"- Epoch: {epoch+1}/{num_epochs}" + Style.RESET_ALL)
        print(Fore.RED + f"- Train Loss: {train_loss:.9f} | Validation Loss: {val_loss:.9f}" + Style.RESET_ALL, end=' ')
        print(Fore.CYAN + f"| Current Learning Rate: {scheduler.optimizer.param_groups[0]['lr']:.7f}" + Style.RESET_ALL, end=' ')
        print(Fore.GREEN + f"| NMSE: {avg_nmse:.7f}" + Style.RESET_ALL)
        
        # Saving Models Checkpoint
        if avg_nmse<=min(nmse_score):
            params = {'epoch': epoch+1,
                      'model': model.state_dict(),
                      'optimizer': optimizer.state_dict(),
                      'scheduler': scheduler.state_dict()}
            
            # Saving Trained Weight
            if (epoch+1)<=400:
                torch.save(params, f'model400.pth')
            else:
                torch.save(params, f'model1000.pth')  
                
                
    np.savetxt(f'nmse_scores.csv', np.array(nmse_score), delimiter=",")
    np.savetxt(f'train_losses.csv', np.array(train_losses), delimiter=",")
    np.savetxt(f'val_losses.csv', np.array(val_losses), delimiter=",")

Model Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Mini Batch Training:   0%|          | 0/500 [00:00<?, ?it/s]

Validating The Model:   0%|          | 0/150 [00:00<?, ?it/s]

Calculating NMSE:   0%|          | 0/100 [00:00<?, ?it/s]

- Epoch: 1/1000
- Train Loss: 0.000645956 | Validation Loss: 0.000454863 | Current Learning Rate: 0.0020000 | NMSE: 0.0336712


...

- Epoch: 999/1000
- Train Loss: 0.000000175 | Validation Loss: 0.000000222 | Current Learning Rate: 0.0000500 | NMSE: -33.6221790


Mini Batch Training:   0%|          | 0/500 [00:00<?, ?it/s]

Validating The Model:   0%|          | 0/150 [00:00<?, ?it/s]

Calculating NMSE:   0%|          | 0/100 [00:00<?, ?it/s]

- Epoch: 1000/1000
- Train Loss: 0.000000175 | Validation Loss: 0.000000223 | Current Learning Rate: 0.0000500 | NMSE: -33.6163770
